<a href="https://colab.research.google.com/github/aryanshenoy02/Banking-Fraud-Detection/blob/main/notebooks/autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

# uploaded is a dict like {'whatever_name.csv': b'...bytes...'}
# next(iter(...)) grabs the first (and only) key, i.e. the filename
filename = next(iter(uploaded))

try:
    # Skipping the first 3 rows of metadata to start directly from the table headers
    df = pd.read_csv(filename, skiprows=3)
    print(f"Successfully loaded '{filename}' by skipping the metadata rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading CSV: {e}")

In [ ]:
combined = df['Date'].astype(str) + ' ' + df['Time'].astype(str)


parsed = pd.to_datetime(
    combined.str.replace(r'\bSept\b', 'Sep', regex=True),
    format='%b %d, %Y %I:%M %p', #example - November 15, 2026 09:30 AM
    errors='coerce' #to replace unparseable data with NaT (Not a Time)
)

invalid_rows = df[parsed.isna()]

display(invalid_rows)

#boolean filtering to remove the invalid rows
df = df[parsed.notna()].copy()
#creating Datetime with no NaN values
df['Datetime'] = parsed[parsed.notna()]

In [ ]:

# 1. Drop rows where all columns are NaN (if any exist at the end of the file)
df = df.dropna(how='all').copy()

# 2. Clean the 'Transaction Details' column
# We remove case-insensitive variations of "Paid to " and "Received from "
df['Transaction Details'] = df['Transaction Details'].str.replace(r'^(Paid to|Received from)\s+', '', case=False, regex=True)

# Sort chronologically so we can calculate the time difference forward in time
df = df.sort_values('Datetime').reset_index(drop=True)

# 4. Calculate the time difference between consecutive transactions (in hours)
df['Time_Diff_Hours'] = df['Datetime'].diff().dt.total_seconds() / 3600.0
df['Time_Diff_Hours'] = df['Time_Diff_Hours'].fillna(0)

# Display the cleaned and processed DataFrame
display(df[['Transaction Details', 'Transaction Type', 'Amount', 'Time_Diff_Hours']].head(10))

In [ ]:
import numpy as np

df['Counterparty_Transaction_Count'] = df.groupby('Transaction Details').cumcount()
df['hour'] = df['Datetime'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dayofweek'] = df['Datetime'].dt.dayofweek
df['is_new_counterparty'] = (df['Counterparty_Transaction_Count'] == 0).astype(int)
df['is_debit'] = (df['Transaction Type'] == 'DEBIT').astype(int)

In [ ]:
import re

def is_known_merchant(name):
    # Strip whitespace
    name = str(name).strip()

    # Pattern: looks like a raw ID — e.g. KA01AR2800
    # Characteristics: no spaces, mix of uppercase letters and digits, length < 15
    raw_id_pattern = r'^[A-Z0-9]{5,15}$'

    if re.match(raw_id_pattern, name):
        return 0  # Unknown — raw ID
    else:
        return 1  # Known — readable merchant name

df['is_known_merchant'] = df['Transaction Details'].apply(is_known_merchant)

In [ ]:
df_ = df.drop(columns = ['Date','Time','Transaction ID','UTR','Transaction Type','Credit/debit instrument','Datetime','hour','Transaction Details'])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

cols_to_scale = ['Amount', 'Time_Diff_Hours', 'Counterparty_Transaction_Count']

scaler = MinMaxScaler()
df_[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers

class AnomalyDetector(Model):
  def __init__(self):
    super(AnomalyDetector, self).__init__()
    self.encoder = tf.keras.Sequential([
        layers.Dense(16, activation='relu'),
        layers.Dense(8, activation='relu'),
        layers.Dense(4, activation='relu')
    ])

    self.decoder = tf.keras.Sequential([
        layers.Dense(8, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(input_dim, activation='linear')
    ])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded

input_dim = df_.shape[1]
autoencoder = AnomalyDetector()
autoencoder.compile(optimizer='adam', loss='mse')

history = autoencoder.fit(
    df_, df_,
    epochs=50,
    batch_size=32,
    shuffle=True,
)

In [ ]:
import numpy as np

# Get reconstructed output
reconstructions = autoencoder.predict(df_)

# Calculate MSE per transaction (mean across all 9 features)
mse = np.mean(np.power(df_ - reconstructions, 2), axis=1)

In [ ]:
threshold = np.percentile(mse, 99)  # top 1% flagged as anomalies
print(f"Threshold: {threshold:.4f}")

df['reconstruction_error'] = mse
df['is_anomaly'] = (mse > threshold).astype(int)

# See what got flagged
print(df[df['is_anomaly'] == 1][['Transaction Details', 'Amount', 'reconstruction_error']].sort_values('reconstruction_error', ascending=False))

In [ ]:
def compute_trust_multiplier(row, contacts):
    multiplier = 1.0

    # Known contact — strong trust signal
    if row['Transaction Details'] in contacts:
        multiplier *= 0.5

    if row['Transaction Details'] in known_corp:
        multiplier *= 0.8

    # Known merchant — moderate trust signal
    if row['is_known_merchant'] == 1:
        multiplier *= 0.7

    return multiplier

known_corp = [
    "AMAZON"
]
contacts = [
    "BHOOMIKA SRIDHAR",
    "NAGARAJU",
    "Mr Yuvan R",
    "BirthGiver(Amma)",
    "SWIGGY",
    "MYNTRA"
    ""
]

df['trust_multiplier'] = df.apply(
    lambda row: compute_trust_multiplier(row, contacts), axis=1
)

df['final_anomaly_score'] = df['reconstruction_error'] * df['trust_multiplier']

# Reapply threshold on final score
threshold_final = np.percentile(df['final_anomaly_score'], 90)
df['is_anomaly'] = (df['final_anomaly_score'] > threshold_final).astype(int)
print(df[df['is_anomaly'] == 1][['Transaction Details', 'Amount', 'reconstruction_error']].sort_values('reconstruction_error', ascending=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(mse, bins=50)
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Count')
plt.title('Transaction Reconstruction Error Distribution')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.scatter(df['Amount'], df['reconstruction_error'], alpha=0.5)
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold')
plt.xlabel('Amount (scaled)')
plt.ylabel('Reconstruction Error')
plt.title('Reconstruction Error vs Amount')
plt.legend()
plt.show()